Perform power transformations and normalization techniques, including quantile transformation, to stabilize variance and minimize the impact of outliers on downstream machine learning models.

In [1]:
import pandas as pd
import numpy as np

In [2]:
np.random.seed(42)

In [3]:
income = np.random.exponential(scale=5000, size=200)
income[10] = 150000  
income[50] = 200000

In [4]:
df = pd.DataFrame({'income': income})

In [5]:
y = df['income'] * 2 + np.random.normal(0, 1000, 200)

In [6]:
print("First few rows of raw data:")
print(df.head())
print("\nMax income value (Notice the outlier):", df['income'].max())

First few rows of raw data:
         income
0   2346.340450
1  15050.607155
2   6583.728468
3   4564.712769
4    848.124352

Max income value (Notice the outlier): 200000.0


In [7]:
from sklearn.preprocessing import QuantileTransformer

In [8]:
qt = QuantileTransformer(output_distribution='normal', random_state=42)

In [9]:
df['quantile_income'] = qt.fit_transform(df[['income']])

c:\python312\Lib\site-packages\sklearn\preprocessing\_data.py:2905: UserWarning: n_quantiles (1000) is greater than the total number of samples (200). n_quantiles is set to n_samples.
  warnings.warn(


In [10]:
print("Data after quantile transformation:")
print(df[['income', 'quantile_income']].head())
print("\nNew Max Value after transformation:", round(df['quantile_income'].max(), 3))

Data after quantile transformation:
         income  quantile_income
0   2346.340450        -0.170876
1  15050.607155         1.748358
2   6583.728468         0.639323
3   4564.712769         0.261159
4    848.124352        -0.970779

New Max Value after transformation: 5.199


In [11]:
from sklearn.preprocessing import PowerTransformer

In [12]:
pt = PowerTransformer(method='yeo-johnson')

In [13]:
df['power_income'] = pt.fit_transform(df[['income']])

In [14]:
print("Data after Power Transformation:")
print(df[['income', 'power_income']].head())

Data after Power Transformation:
         income  power_income
0   2346.340450     -0.185063
1  15050.607155      1.361925
2   6583.728468      0.642954
3   4564.712769      0.340531
4    848.124352     -0.932285


In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

In [16]:
model = LinearRegression()

In [17]:
score_raw = cross_val_score(model, df[['income']], y, cv=5).mean()

In [18]:
score_qt = cross_val_score(model, df[['quantile_income']], y, cv=5).mean()

In [19]:
score_pt = cross_val_score(model, df[['power_income']], y, cv=5).mean()

In [20]:
print("Cross-Validation Scores (Higher is better):")
print("Raw Data Score:", round(score_raw, 4))
print("Quantile Transformed Score:", round(score_qt, 4))
print("Power Transformed Score:", round(score_pt, 4))

Cross-Validation Scores (Higher is better):
Raw Data Score: 0.9916
Quantile Transformed Score: -0.6167
Power Transformed Score: -1.0684
